# Matriz de confusión — modelo de clasificación de daños

Este notebook carga el modelo ya entrenado (`modelo_classification.keras`), reconstruye el conjunto de validación con el mismo split usado durante el entrenamiento, y genera la matriz de confusión por clase.

## Imports

In [ ]:
import os
import glob
import json
import subprocess

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, multilabel_confusion_matrix, roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

## Descarga de datos desde S3

Si ya tenés la carpeta `data_deteccion/` descargada de una corrida anterior, podés saltear esta celda.

In [ ]:
LOCAL_DATA_DIR = "data_deteccion"
os.makedirs(f"{LOCAL_DATA_DIR}/images", exist_ok=True)
os.makedirs(f"{LOCAL_DATA_DIR}/labels", exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_classification"

subprocess.run(["aws", "s3", "sync", f"{BUCKET}/images/", f"{LOCAL_DATA_DIR}/images/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/labels/", f"{LOCAL_DATA_DIR}/labels/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/classes.txt", f"{LOCAL_DATA_DIR}/classes.txt", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/data.yaml", f"{LOCAL_DATA_DIR}/data.yaml", "--quiet"], check=True)

n_images = len(os.listdir(f"{LOCAL_DATA_DIR}/images"))
n_labels = len(os.listdir(f"{LOCAL_DATA_DIR}/labels"))
print(f"Imagenes: {n_images} | Labels: {n_labels}")

## Clases finales

In [ ]:
with open(f"{LOCAL_DATA_DIR}/classes.txt") as f:
    all_classes = [line.strip() for line in f if line.strip()]

FINAL_CLASSES = [
    "Diagonal crack",
    "Concrete spalling",
    "Corrosion",
    "Shear crack",
    "Flexural/rocking crack",
    "Debris",
    "Detachment crack",
]

orig_id_to_name = {i: name for i, name in enumerate(all_classes)}
name_to_final_idx = {name: idx for idx, name in enumerate(FINAL_CLASSES)}

print(f"{len(all_classes)} clases en el dataset original, nos quedamos con {len(FINAL_CLASSES)}")

## Reconstrucción del dataframe (mismo criterio que en el entrenamiento)

In [ ]:
label_files = sorted(glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"))

rows = []
descartadas = 0

for label_path in label_files:
    img_name = os.path.splitext(os.path.basename(label_path))[0]

    img_path = None
    for ext in (".jpg", ".jpeg", ".png"):
        candidate = f"{LOCAL_DATA_DIR}/images/{img_name}{ext}"
        if os.path.exists(candidate):
            img_path = candidate
            break
    if img_path is None:
        continue

    with open(label_path) as f:
        lines = [l.strip() for l in f if l.strip()]

    presentes = set()
    for line in lines:
        class_id = int(line.split()[0])
        class_name = orig_id_to_name[class_id]
        if class_name in name_to_final_idx:
            presentes.add(name_to_final_idx[class_name])

    if not presentes:
        descartadas += 1
        continue

    multi_hot = np.zeros(len(FINAL_CLASSES), dtype=int)
    for idx in presentes:
        multi_hot[idx] = 1

    rows.append([img_path] + multi_hot.tolist())

df = pd.DataFrame(rows, columns=["image_path"] + FINAL_CLASSES)
print(f"quedaron {len(df)} imagenes, se descartaron {descartadas} por tener solo clases fuera de las finales")
df.head()

## Split de validación (mismo seed que en el entrenamiento)

In [ ]:
X = df["image_path"].values.reshape(-1, 1)
y = df[FINAL_CLASSES].values

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(msss.split(X, y))

val_df = df.iloc[val_idx].reset_index(drop=True)
print(f"val: {len(val_df)}")
print(val_df[FINAL_CLASSES].sum())

## Dataset de validación

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_dataset(dataframe, training):
    paths = dataframe["image_path"].values
    labels = dataframe[FINAL_CLASSES].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(tf.data.AUTOTUNE)

val_ds = make_dataset(val_df, training=False)

## Carga del modelo y de los umbrales por clase

In [ ]:
MODEL_PATH = "models/modelo_classification.keras"
THRESHOLDS_PATH = "models/thresholds.json"

model = tf.keras.models.load_model(MODEL_PATH)

with open(THRESHOLDS_PATH) as f:
    best_thresholds = json.load(f)

best_thresholds

## Predicciones sobre el conjunto de validación

In [ ]:
val_probs = model.predict(val_ds)
val_true = val_df[FINAL_CLASSES].values

val_preds_tuned = np.array([
    (val_probs[:, i] > best_thresholds[cls]).astype(int)
    for i, cls in enumerate(FINAL_CLASSES)
]).T

## Matriz de confusión por clase

In [ ]:
# Nota: a diferencia del modelo de detección (clasificación binaria), este es un
# problema multi-etiqueta: cada imagen puede pertenecer a mas de una clase a la vez.
# Por eso no existe una unica matriz de confusion NxN, sino una matriz binaria
# (presente / ausente) por cada clase, evaluada de forma independiente.

print(classification_report(val_true, val_preds_tuned, target_names=FINAL_CLASSES, zero_division=0))

mcm = multilabel_confusion_matrix(val_true, val_preds_tuned)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (cls, cm) in enumerate(zip(FINAL_CLASSES, mcm)):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No", "Sí"], yticklabels=["No", "Sí"],
                ax=axes[i], cbar=False)
    axes[i].set_title(cls, fontsize=10)
    axes[i].set_xlabel("Predicción")
    axes[i].set_ylabel("Real")

# quedan 8 subplots para 7 clases, el ultimo lo ocultamos
axes[-1].axis("off")

plt.suptitle("Matrices de confusión por clase: CNN clasificación de daños", y=1.02)
plt.tight_layout()
plt.savefig("matriz_confusion_clasificacion.png", dpi=200, bbox_inches="tight")
plt.show()

roc_auc = roc_auc_score(val_true, val_probs, average="macro")
print(f"ROC-AUC (macro): {roc_auc:.4f}")

print("""
Precision: de todo lo que el modelo predijo como una clase, qué porcentaje
           realmente era esa clase (mide falsas alarmas).
Recall:    de todo lo que realmente era esa clase, qué porcentaje el modelo
           detectó correctamente (mide casos no detectados / falsos negativos).
F1-score:  balance entre precision y recall en un solo número.
ROC-AUC:   qué tan bien separa el modelo cada clase del resto, probando
           todos los umbrales de decisión posibles (no solo el ajustado por clase).
           1.0 = separación perfecta | 0.5 = equivalente a azar.
           Acá se reporta como promedio macro entre las 7 clases.
""")